Import Model Configuration
==========================

In [2]:
import yaml

model_cfg = "maskrcnn_resnet50_fpn_supervised.yaml"

with open("./configs/" + model_cfg , "r") as f:
    cfg = yaml.safe_load(f)

cfg['experiment_name']

'maskrcnn_resnet50_fpn_supervised'

Dataset Class
=============

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask
import numpy as np


class COCOSegmentationDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.root_dir = root_dir
        self.transforms = transforms

        # path to the JSON annotation file
        self.ann_path = os.path.join(root_dir, "_annotations.coco.json")
        self.coco = COCO(self.ann_path)
        self.ids = list(self.coco.imgs.keys())

    def __getitem__(self, index):
        coco = self.coco
        img_id = self.ids[index]
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.root_dir, img_info['file_name'])

        img = read_image(img_path)  # shape: [C, H, W]
        img = tv_tensors.Image(img)
        h, w = img.shape[-2:]

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        masks = []
        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            if 'segmentation' not in ann or not ann['segmentation']:
                continue  # Skip if missing or empty

            try:
                rles = coco_mask.frPyObjects(ann['segmentation'], h, w)
                mask = coco_mask.decode(rles)
            except Exception as e:
                print(f"[!] Skipping bad annotation in image {img_path}: {e}")
                continue
            if mask.ndim == 3:
                mask = mask.any(axis=2)
            mask = torch.as_tensor(mask, dtype=torch.uint8)
            masks.append(mask)

            x, y, bw, bh = ann['bbox']
            boxes.append(torch.tensor([x, y, x + bw, y + bh], dtype=torch.float32))
            labels.append(ann.get('category_id', 1))
            areas.append(ann['area'])
            iscrowd.append(ann.get('iscrowd', 0))

        if masks:
            masks = torch.stack(masks)
            boxes = torch.stack(boxes)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)
        else:
            masks = torch.zeros((0, h, w), dtype=torch.uint8)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(h, w)),
            "masks": tv_tensors.Mask(masks),
            "labels": labels,
            "image_id": img_id,
            "area": areas,
            "iscrowd": iscrowd
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

            # ⚠️ Filter out zero-area boxes *after transforms*
            boxes = target["boxes"]
            keep = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])

            # Apply keep mask to all fields
            target["boxes"] = tv_tensors.BoundingBoxes(boxes[keep], format="XYXY", canvas_size=img.shape[-2:])
            target["labels"] = target["labels"][keep]
            if "masks" in target:
                target["masks"] = target["masks"][keep]
            if "area" in target:
                target["area"] = target["area"][keep]
            if "iscrowd" in target:
                target["iscrowd"] = target["iscrowd"][keep]

        return img, target

    def __len__(self):
        return len(self.ids)


Augmentiations
=============

In [ ]:
from torchvision.transforms import v2 as T
import torch

def get_transform(train: bool):
    if train:
        transforms = [
            T.RandomPhotometricDistort(),  # Color jitter
            T.Resize((512, 512)),          # Fixed resize for stability
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10, fill=0, expand=False),  # Limit rotation area
            # T.RandomZoomOut(fill=0, p=0.3),  # Optional — leave off for now
            # T.RandomCrop((512, 512), pad_if_needed=False, fill=0),  # Only if images are bigger
        ]
    else:
        transforms = [
            T.Resize((512, 512)),                                   # Consistent resize for val/test
        ]

    transforms += [
        T.ToDtype(torch.float32, scale=True),                       # Convert image to float32 in [0, 1]
        T.ToPureTensor(),                                           # Convert to torch.Tensor (Image, Mask, BBoxes)
    ]

    return T.Compose(transforms)

Log Hyperparams to TensorBoard
===============================

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import shutil, os

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

log_dir = f"runs/{cfg['experiment_name']}_{timestamp}"

if os.path.exists(log_dir):
    shutil.rmtree(log_dir)

# Save cfg as hyperparameters (flatten dictionary)
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

writer = SummaryWriter(log_dir=f"runs/{cfg['experiment_name']}_{timestamp}")

# Option 1: Save flattened cfg to TensorBoard as hparams
flattened_cfg = flatten_dict(cfg)

# Option 2: Also log as text (for easier reading)
cfg_text = yaml.dump(cfg, sort_keys=False)
writer.add_text("config", f"```yaml\n{cfg_text}\n```")


Initialization
=============

In [ ]:
import utils  # Your helper functions (make sure it includes `collate_fn` from PyTorch references)
import torch
from torchvision.transforms import v2 as T
from engine import train_one_epoch, evaluate


from models import MODEL_REGISTRY



device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


dataset = COCOSegmentationDataset(
    root_dir="../../../datasets/SupervisedDataset/train",
    transforms=get_transform(train=True)
)

dataset_valid = COCOSegmentationDataset(
    root_dir="../../../datasets/SupervisedDataset/val",
    transforms=get_transform(train=False)
)

dataset_test = COCOSegmentationDataset(
    root_dir="../../../datasets/SupervisedDataset/test",
    transforms=get_transform(train=False)
)


print("Train size:", len(dataset))
print("Validation size:", len(dataset_valid))
print("Test size:", len(dataset_test))

# Dynamically assign number of workers
n_workers = 4 if torch.cuda.is_available() else 0

# Dataloaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=cfg["train"]["batch_size"],
    shuffle=True,
    num_workers = n_workers,
    collate_fn=utils.collate_fn
)

data_loader_valid = torch.utils.data.DataLoader(
    dataset_valid,
    batch_size=1,
    shuffle=False,
    num_workers=n_workers,
    collate_fn=utils.collate_fn
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    num_workers=n_workers,
    collate_fn=utils.collate_fn
)

# Load model
model = MODEL_REGISTRY[cfg["model"]](num_classes=cfg["num_classes"])

# Load pretrained weights from synthetic training
model.load_state_dict(torch.load(
    "outputs/maskrcnn_resnet50_fpn_20250609_1525/model_best.pth",
    map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")
))

# Move to device
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

loading annotations into memory...
Done (t=0.25s)
creating index...
index created!
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Train size: 170
Validation size: 48
Test size: 26


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [ ]:
import csv
import time
import sys
import os
import torch
from tqdm import tqdm

# Optimizer and scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=cfg["train"]["lr"],
    momentum=cfg["train"]["momentum"],
    weight_decay=cfg["train"]["weight_decay"],
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=cfg["train"]["lr_step_size"],
    gamma=cfg["train"]["gamma"],
)

# Output & logging
cfg["output_dir"] = f"outputs/{cfg['experiment_name']}_{timestamp}"
os.makedirs(cfg["output_dir"], exist_ok=True)

csv_path = os.path.join(cfg["output_dir"], "segm_metrics.csv")
best_val = -float("inf")
num_epochs = cfg["train"]["epochs"]

print("Start supervised fine-tuning...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    sys.stdout.flush()
    start_time = time.time()

    model.train()
    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr", utils.SmoothedValue(window_size=1, fmt="{value:.6f}"))

    header = f"Epoch: [{epoch}]"
    warmup_scheduler = None

    if epoch == 0:
        warmup_factor = 1.0 / 1000
        warmup_iters = min(1000, len(data_loader) - 1)
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=warmup_factor, total_iters=warmup_iters
        )

    for batch_idx, (images, targets) in enumerate(tqdm(data_loader, desc=header)):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

        with torch.amp.autocast(device_type='cuda', enabled=False):  # AMP optional
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        if not torch.isfinite(losses):
            print(f"[!] Loss is {losses.item()}, stopping training")
            print(loss_dict)
            sys.exit(1)

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        if warmup_scheduler:
            warmup_scheduler.step()

        metric_logger.update(loss=losses.item(), **loss_dict)
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])

        global_step = epoch * len(data_loader) + batch_idx
        writer.add_scalar("Train/Loss", losses.item(), global_step)
        writer.add_scalar("Train/LR", optimizer.param_groups[0]["lr"], global_step)

    lr_scheduler.step()
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} completed in {epoch_time:.2f} seconds.")

    # === Validation ===
    print("Evaluating on validation set:")
    coco_evaluator = evaluate(model, data_loader_valid, device=device)
    segm_eval = coco_evaluator.coco_eval["segm"]
    segm_stats = segm_eval.stats

    if epoch == 0 and not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow([
                "epoch",
                "AP@[IoU=0.50:0.95]", "AP@0.50", "AP@0.75",
                "AP_small", "AP_medium", "AP_large",
                "AR@max=1", "AR@max=10", "AR@max=100",
                "AR_small", "AR_medium", "AR_large"
            ])

    with open(csv_path, "a", newline="") as f:
        csv_writer = csv.writer(f)
        csv_writer.writerow([epoch] + list(segm_stats))

    # TensorBoard logging
    writer.add_scalar("Val/segm_mAP", segm_stats[0], epoch)
    writer.add_scalar("Val/segm_mAP_50", segm_stats[1], epoch)
    writer.add_scalar("Val/segm_mAP_75", segm_stats[2], epoch)
    writer.add_scalar("Val/segm_mAP_small", segm_stats[3], epoch)
    writer.add_scalar("Val/segm_mAP_medium", segm_stats[4], epoch)
    writer.add_scalar("Val/segm_mAP_large", segm_stats[5], epoch)
    writer.add_scalar("Val/segm_AR_max1", segm_stats[6], epoch)
    writer.add_scalar("Val/segm_AR_max10", segm_stats[7], epoch)
    writer.add_scalar("Val/segm_AR_max100", segm_stats[8], epoch)
    writer.add_scalar("Val/segm_AR_small", segm_stats[9], epoch)
    writer.add_scalar("Val/segm_AR_medium", segm_stats[10], epoch)
    writer.add_scalar("Val/segm_AR_large", segm_stats[11], epoch)

    current_val = segm_stats[0]
    if current_val > best_val:
        best_val = current_val
        torch.save(model.state_dict(), os.path.join(cfg["output_dir"], "model_best.pth"))
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        }, os.path.join(cfg["output_dir"], "checkpoint_best.pth"))

print("\n✅ Supervised fine-tuning completed!")


Start supervised fine-tuning...

Epoch 1/10


Epoch: [0]: 100%|██████████| 85/85 [00:49<00:00,  1.72it/s]


Epoch 1 completed in 49.34 seconds.
Evaluating on validation set:
creating index...
index created!
Test:  [ 0/48]  eta: 0:00:23  model_time: 0.3285 (0.3285)  evaluator_time: 0.0300 (0.0300)  time: 0.4958  data: 0.1348  max mem: 2175
Test:  [47/48]  eta: 0:00:00  model_time: 0.1499 (0.1538)  evaluator_time: 0.0376 (0.0307)  time: 0.1894  data: 0.0041  max mem: 2175
Test: Total time: 0:00:09 (0.1934 s / it)
Averaged stats: model_time: 0.1499 (0.1538)  evaluator_time: 0.0376 (0.0307)
Accumulating evaluation results...
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.01s).
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.765
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.965
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.939
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.012
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0

Epoch: [1]:  44%|████▎     | 37/85 [00:22<00:28,  1.69it/s]Exception in thread Thread-4:
Traceback (most recent call last):
  File "/home/jotac431/miniconda3/envs/coco-env/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/jotac431/miniconda3/envs/coco-env/lib/python3.10/site-packages/tensorboard/summary/writer/event_file_writer.py", line 244, in run
    self._run()
  File "/home/jotac431/miniconda3/envs/coco-env/lib/python3.10/site-packages/tensorboard/summary/writer/event_file_writer.py", line 275, in _run
    self._record_writer.write(data)
  File "/home/jotac431/miniconda3/envs/coco-env/lib/python3.10/site-packages/tensorboard/summary/writer/record_writer.py", line 40, in write
    self._writer.write(header + header_crc + data + footer_crc)
  File "/home/jotac431/miniconda3/envs/coco-env/lib/python3.10/site-packages/tensorboard/compat/tensorflow_stub/io/gfile.py", line 775, in write
    self.fs.append(self.filename, file_content, self.binary_m

FileNotFoundError: [Errno 2] No such file or directory: b'runs/maskrcnn_resnet50_fpn_supervised_20250619_1134/events.out.tfevents.1750329290.DESKTOP-KGSJ5S1.255333.0'